# Phase 13 Part 1A-R1 — Supervisor Protocol-Freeze Amendment

**Model:** Gemini 3.8 Flash Medium · **Date:** 2026-09-21
**Manifest:** `phase4_v3` · **Taxonomy:** `kub_size_clean_v3_v1`
**Part 1A archaeology/reconstruction:** ACCEPTED
**Part 1A scientific protocol:** NOT YET frozen for execution — **Part 1A-R1 ONLY is authorized**

---

> **AMENDMENT PHASE ONLY.** This notebook records exact supervisor decisions, amends the
> clean-v3 Phase-13 protocol candidate, and produces the final supervisor-frozen protocol
> artifact.
>
> **NO TRAINING · NO OPTIMIZER · NO MODEL FORWARD PASS · NO LOCALIZER INFERENCE ·
> NO PREDICTED-REGION INFERENCE · NO PROPOSALS · NO TEST ACCESS · NO PART 1B · NO PHASE 14.**
>
> Authorized reads: already-frozen clean-v3 VAL predictions/tables, used **only** to derive
> the prospectively defined hard-set membership (§17).

## Companion-notebook rule

The accepted Part-1A notebook is **NOT modified** by this task:

- `phase13_clean_v3_localizer_protocol_freeze.ipynb`
- SHA-256 `68f787e1f70fcf706442b342e62cc615eeb347653e753149987becf793ac2bd0`

This notebook is a small companion amendment.


In [1]:
import sys, json
from pathlib import Path
import pandas as pd
pd.set_option('display.max_rows', None); pd.set_option('display.max_columns', None)
pd.set_option('display.width', None); pd.set_option('display.max_colwidth', 90)

PROJECT_ROOT = Path(r'D:\Softfors\jup')
CODE_DIR = PROJECT_ROOT / 'notebooks/phasewise/phase_13_predicted_roi_localization/code'
for p in (str(PROJECT_ROOT), str(CODE_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

import phase13_localizer_freeze_amendment as r1
TABLES = PROJECT_ROOT / 'reports/tables'
AUDITS = PROJECT_ROOT / 'reports/audits'

# Read-only regeneration: derives hard sets from frozen clean-v3 tables and writes the
# frozen protocol artifact. No model is loaded, no forward pass, no optimizer, no inference.
hashes = r1.write_all(write_frozen=True)
print(f'wrote {len(hashes)} artifacts')
print('training 0 | optimizer steps 0 | forward passes 0 | localizer inference 0 | '
      'predicted-region inference 0 | proposals 0 | TEST reads 0')


wrote 5 artifacts
training 0 | optimizer steps 0 | forward passes 0 | localizer inference 0 | predicted-region inference 0 | proposals 0 | TEST reads 0


## 1. Accepted Part-1A notebook — unchanged verification

The accepted Part-1A notebook SHA must still match. This is checked before anything else.


In [2]:
nb_path = r1.PART1A_NOTEBOOK
actual = r1.sha256_file(nb_path)
print('path    :', nb_path.relative_to(PROJECT_ROOT))
print('expected:', r1.PART1A_NOTEBOOK_SHA)
print('actual  :', actual)
print('UNCHANGED:', actual == r1.PART1A_NOTEBOOK_SHA)
assert actual == r1.PART1A_NOTEBOOK_SHA, 'accepted Part-1A notebook was modified'


path    : notebooks\phasewise\phase_13_predicted_roi_localization\phase13_clean_v3_localizer_protocol_freeze.ipynb
expected: 68f787e1f70fcf706442b342e62cc615eeb347653e753149987becf793ac2bd0
actual  : 68f787e1f70fcf706442b342e62cc615eeb347653e753149987becf793ac2bd0
UNCHANGED: True


## 2. Supervisor decisions (verbatim dispositions)

| § | Decision | Disposition |
|---|---|---|
| §1 | Hard sets | re-derive on clean-v3 using the **same historical definition**, freeze **before** training |
| §2 | Heatmap resolution | **APPROVED MATERIAL CORRECTION** — explicit `heatmap_resolution=128` |
| §3 | Stage-4D `clean_v3_results` label | `STAGE4D_DESIGN_AUDIT_LABEL_PROVENANCE_DEFECT` → `HISTORICAL_v2_PROVENANCE_ONLY` |
| §4 | Missing `val_search_epoch079.csv` | `NON_BLOCKING_HISTORICAL_ARTIFACT_GAP` — not fabricated |
| §5 | Proposal grid | clean-v3 primary grid search **DISABLED**; historical 20-grid `DEFERRED_SECONDARY_SENSITIVITY_ONLY` |
| §6 | Checkpoint selection | fixed-config historical lexicographic score |
| §7 | Coverage | historical `covered90 >= 0.90` preserved; no new post-training threshold |
| §8 | Phase-14 distribution mismatch | `DEFERRED_REQUIREMENT_FOR_PHASE14_IF_GATE_ALLOWS` — **not** a Phase-13 change |
| §9 | Downstream geometry | frozen GT-free config, margin **0.20** (not the Phase-12 oracle 0.25) |
| §10 | Downstream composition | `PREDICTED_REGION_FLOAT_GLOBAL_RETAINED` |
| §11 | Downstream model scope | all four; no reference model |
| §12 | Retention contract | ratio only where `oracle_gain > 0`; no sufficiency threshold |
| §13 | Gate | dimension B `CANDIDATE_FOR_SUPERVISOR_FREEZE`; gate `NOT_DECIDED` |


## 3. Clean-v3 hard-set derivation

Both cohorts are re-derived from **already-frozen clean-v3 VAL tables** using the exact
historical definition reconstructed in Part 1A. No segmentation inference is run.

**`hard_tiny`** — TINY native components whose clean-v3 UNet++ GLOBAL prediction misses them:

> `global_detected = at least 1 native-space pixel of the 8-connected GT component has`
> `GLOBAL_NATIVE probability >= 0.5` (executing definition, `phase12_oracle_inference.py`
> `component_detection` / driver line 459)

**`global_empty_image`** — FULL_FIELD images whose clean-v3 UNet++ GLOBAL prediction is empty
(`prediction_empty == true`), cross-validated against the frozen Phase-11 clean-v3 table.


In [3]:
ht = pd.read_csv(r1.HARD_TINY_OUT)
print('CLEAN-v3 HARD TINY SET — count:', len(ht))
print('source      :', ht['clean_v3_source'].iloc[0])
print('source SHA  :', ht['source_sha'].iloc[0])
print('supporting  :', ht['supporting_source'].iloc[0])
print('anchor      :', ht['anchor_model'].iloc[0], '/', ht['anchor_role'].iloc[0])
print('all TINY and undetected:',
      bool((ht['size_stratum'] == 'TINY').all() and (~ht['global_detected'].astype(bool)).all()))
display(ht[['sample_id', 'native_component_id', 'size_stratum', 'global_detected']])


CLEAN-v3 HARD TINY SET — count: 15
source      : reports/tables/phase12_clean_v3_oracle_component_metrics.csv
source SHA  : d66933a3c2821d3ab9375eadb5c0540fea145edcf49b648064fe9aa4c460c753
supporting  : reports/tables/phase12_clean_v3_oracle_geometry.csv
anchor      : unetpp / HISTORICAL_PHASE13_HARDSET_REPLICATION_ANCHOR_ONLY
all TINY and undetected: True


,sample_id,native_component_id,size_stratum,global_detected
0,2286db1a7e6b8867cde46a4294ba0c10,2,TINY,False
1,416e6f7d9e35c6bf4bde51e191a7e92d,1,TINY,False
2,416e6f7d9e35c6bf4bde51e191a7e92d,2,TINY,False
3,5779b9fca1dec2e3598ccdcbad32608f,2,TINY,False
4,6a5d5c051d6c898d98c90422da190bf0,2,TINY,False
5,8c8d2107815002b7913bd6a42f93b394,1,TINY,False
6,8c8d2107815002b7913bd6a42f93b394,2,TINY,False
7,9f367e7db192bfe532ace746fd788fb9,2,TINY,False
8,a626dd389032147c7d4ebd2b1db39e45,1,TINY,False
9,ad4af1ce1b3dde6ad04fa095aacd92f5,1,TINY,False


In [4]:
ge = pd.read_csv(r1.GLOBAL_EMPTY_OUT)
print('CLEAN-v3 GLOBAL-EMPTY SET — count:', len(ge))
print('source      :', ge['clean_v3_source'].iloc[0])
print('source SHA  :', ge['source_sha'].iloc[0])
print('supporting  :', ge['supporting_source'].iloc[0])
print('cross-validated Phase-11 vs Phase-12: True (asserted during derivation)')
display(ge[['sample_id', 'prediction_empty', 'anchor_model', 'anchor_role']])


CLEAN-v3 GLOBAL-EMPTY SET — count: 5
source      : reports/tables/phase12_clean_v3_oracle_case_metrics.csv
source SHA  : 080f158fb8b3e3552ac7854c6d9a116e11564bcefa9efdcdbb3511dba090b436
supporting  : reports/tables/phase11_clean_v3_per_model_case_failure_metrics.csv
cross-validated Phase-11 vs Phase-12: True (asserted during derivation)


,sample_id,prediction_empty,anchor_model,anchor_role
0,416e6f7d9e35c6bf4bde51e191a7e92d,True,unetpp,HISTORICAL_PHASE13_HARDSET_REPLICATION_ANCHOR_ONLY
1,5dd4436a2065b8855ce1dfbed228bce7,True,unetpp,HISTORICAL_PHASE13_HARDSET_REPLICATION_ANCHOR_ONLY
2,8c8d2107815002b7913bd6a42f93b394,True,unetpp,HISTORICAL_PHASE13_HARDSET_REPLICATION_ANCHOR_ONLY
3,ad4af1ce1b3dde6ad04fa095aacd92f5,True,unetpp,HISTORICAL_PHASE13_HARDSET_REPLICATION_ANCHOR_ONLY
4,b0473630de9211114c94c7dc0363ebd9,True,unetpp,HISTORICAL_PHASE13_HARDSET_REPLICATION_ANCHOR_ONLY


In [5]:
print('historical v2 counts (NOT a target):', {'hard_tiny': 16, 'global_empty': 7})
print('clean-v3 re-derived counts        :', {'hard_tiny': len(ht), 'global_empty': len(ge)})
print()
print('UNet++ role:', r1.HARDSET_ANCHOR_ROLE)
print('UNet++ is NOT:', ', '.join(r1.HARDSET_ANCHOR_IS_NOT))
print('reference model selected:', r1.DOWNSTREAM_REFERENCE_MODEL)
print()
print('These replication cohorts may participate in the historical-style checkpoint metric.')
print('They MUST NOT replace the architecture-independent frozen gate cohorts:')
for c in r1.build_frozen_protocol(ht, ge, 'x', 'x')['hard_sets']['must_not_replace_gate_cohorts']:
    print('  -', c)


historical v2 counts (NOT a target): {'hard_tiny': 16, 'global_empty': 7}
clean-v3 re-derived counts        : {'hard_tiny': 15, 'global_empty': 5}

UNet++ role: HISTORICAL_PHASE13_HARDSET_REPLICATION_ANCHOR_ONLY
UNet++ is NOT: winner, best model, Phase-13 downstream reference model, final architecture choice
reference model selected: None

These replication cohorts may participate in the historical-style checkpoint metric.
They MUST NOT replace the architecture-independent frozen gate cohorts:
  - Phase-11 ALL_FOUR_LOW
  - Phase-11 hard cohort
  - Phase-11 disagreement cohort
  - Phase-12 FULL_FIELD 103


## 4. Heatmap resolution contract

The historical helper defect is **not** silently used. Every clean-v3 path must pass
`heatmap_resolution=128` explicitly, with fail-loud shape assertions.


In [6]:
hc = r1.HEATMAP_CONTRACT
print('input          :', hc['input'])
print('output heatmap :', hc['output_heatmap'])
print('heatmap_resolution:', hc['heatmap_resolution'], '(explicit, never a constructor default)')
print('applies to     :', ', '.join(hc['applies_to']))
print('fail-loud assertions:')
for a in hc['fail_loud_assertions']:
    print('  -', a)
print()
print('defect       :', hc['historical_helper_defect'])
print('preferred fix:', hc['preferred_remediation'], '->', hc['remediation_options'][hc['preferred_remediation']])
print('historical frozen outputs must not be modified:', hc['historical_frozen_outputs_must_not_be_modified'])
print()
print('Any violation -> STOP / CHECK_REQUIRED')


input          : 512x512
output heatmap : 128x128
heatmap_resolution: 128 (explicit, never a constructor default)
applies to     : training, validation, checkpoint reproduction, proposal inference
fail-loud assertions:
  - model output spatial shape == (128, 128)
  - target heatmap spatial shape == (128, 128)
  - prediction and target shapes identical

defect       : src/inference/learned_roi.predict_localizer omits heatmap_resolution=128
preferred fix: B -> create a clean-v3-safe inference wrapper (PREFERRED — avoids any risk of altering historical reproduction)
historical frozen outputs must not be modified: True

Any violation -> STOP / CHECK_REQUIRED


## 5. Frozen proposal configuration

The historically selected configuration is known **before** clean-v3 localizer results are
inspected, so it is frozen now. Letting the clean-v3 VAL cohort choose another configuration
would introduce a new VAL-dependent post-processing optimization.


In [7]:
cfg = r1.FIXED_PROPOSAL_CONFIG
print('FROZEN clean-v3 primary proposal configuration:')
for k, v in cfg.items():
    print(f'  {k:10s} = {v}')
print()
print('provenance:', r1.FIXED_PROPOSAL_CONFIG_PROVENANCE)
print('clean-v3 primary grid search :', 'DISABLED')
print('historical 20-grid           : DEFERRED_SECONDARY_SENSITIVITY_ONLY')
print('  may only run later under separate authorization, after the primary checkpoint,')
print('  primary proposal config and primary result are frozen.')


FROZEN clean-v3 primary proposal configuration:
  threshold  = 0.1
  margin     = 0.2
  top_k      = 5
  nms_iou    = 0.3

provenance: VERIFIED_HISTORICAL_SELECTION + PROSPECTIVELY_FROZEN_FOR_CLEAN_V3_PRIMARY
clean-v3 primary grid search : DISABLED
historical 20-grid           : DEFERRED_SECONDARY_SENSITIVITY_ONLY
  may only run later under separate authorization, after the primary checkpoint,
  primary proposal config and primary result are frozen.


## 6. Checkpoint-selection contract

Lexicographic evidence priorities, computed at **one fixed proposal configuration only**.

### 6.1 Provenance disclosure (recorded, not silently asserted)

The supervisor R1 contract specifies an **8-term** tuple described as "reconstructed from
executing code". Verification of the executing code shows a **4-term** tuple. The deviation is
recorded explicitly so it is never mis-stated as historical reproduction.


In [8]:
ex = r1.HISTORICAL_EXECUTING_SELECTION
print('EXECUTING CODE (authority rank 1):')
print('  source :', ex['source'], 'line', ex['source_line'])
print('  score  :', ex['score_tuple'])
print('  length :', ex['tuple_length'])
print('  improvement:', ex['improvement_rule'], '| ties:', ex['ties'], '| patience:', ex['patience'])
print('  proposal config used:', ex['proposal_config_used'])
print('  checkpoint key:', ex['checkpoint_key'])
print('  note:', ex['note'])
print()
print('FROZEN PROTOCOL DOC (PHASE13B.md, rank 2): length', r1.HISTORICAL_PROTOCOL_DOC_SELECTION['tuple_length'])
print('STAGE-3 CORRECTION DOC (rank 2)          : length', r1.HISTORICAL_STAGE3_SELECTION['tuple_length'])
print()
cs = r1.CLEAN_V3_CHECKPOINT_SELECTION
print('CLEAN-v3 SUPERVISOR-FROZEN CONTRACT (authoritative):')
for i, t in enumerate(cs['tuple_ordering'], start=1):
    print(f'  {i}. {t}')
print('  evaluated at:', cs['evaluated_at'])
print('  hard-set terms use:', cs['hard_tiny_and_global_empty_terms_use'])
print('  zero denominator ->', cs['zero_denominator_rule'])
print('  per-epoch proposal grid search:', cs['per_epoch_proposal_grid_search'])
print()
print('DISCLOSURE:', cs['provenance_disclosure'])


EXECUTING CODE (authority rank 1):
  source : scripts/training/train_phase13b_localizer.py line 164
  score  : (tiny_recall90, overall_recall90, -mean_roi_area_fraction, -val_loss)
  length : 4
  improvement: score > best_score (strict) | ties: bad epoch | patience: 15
  proposal config used: BEST OF THE 20-CONFIG GRID PER EPOCH (VAL-dependent)
  checkpoint key: best_validation_score
  note: The executing code used a 4-tuple that did NOT include hard_tiny_recovered90, global_empty_images_covered90 or multi_component_recall90, even though those were computed and logged into history.csv.

FROZEN PROTOCOL DOC (PHASE13B.md, rank 2): length 8
STAGE-3 CORRECTION DOC (rank 2)          : length 3

CLEAN-v3 SUPERVISOR-FROZEN CONTRACT (authoritative):
  1. hard_tiny_recovered90
  2. global_empty_images_covered90
  3. tiny_recall90
  4. overall_recall90
  5. multi_component_recall90
  6. -mean_roi_area_fraction
  7. -mean_proposals_per_image
  8. -val_loss
  evaluated at: ONE FIXED PROPOSAL CONFI

## 7. Final v2 → clean-v3 protocol diff

The Part-1A base rows plus five explicit amendment rows (A–E). **Unauthorized material
differences must be 0.**


In [9]:
diff = pd.read_csv(r1.DIFF_OUT)
display(diff[['amendment_id', 'field', 'same_or_changed', 'scientifically_material',
              'authorization']])
print('rows:', len(diff), '| amendments:', diff['amendment_id'].value_counts().to_dict())
print('unauthorized material differences:', int(diff['unauthorized_material_difference'].sum()))
assert int(diff['unauthorized_material_difference'].sum()) == 0


,amendment_id,field,same_or_changed,scientifically_material,authorization
0,PART_1A_BASE,manifest_version,CHANGED,False,SUPERVISOR_FROZEN
1,PART_1A_BASE,taxonomy,CHANGED,False,SUPERVISOR_FROZEN
2,PART_1A_BASE,train_population,CHANGED,False,SUPERVISOR_FROZEN
3,PART_1A_BASE,val_population,CHANGED,False,SUPERVISOR_FROZEN
4,PART_1A_BASE,val_size_strata,CHANGED,True,SUPERVISOR_FROZEN
5,PART_1A_BASE,model_architecture,SAME,False,SUPERVISOR_FROZEN
6,PART_1A_BASE,input_channels,SAME,False,SUPERVISOR_FROZEN
7,PART_1A_BASE,input_resolution,SAME,False,SUPERVISOR_FROZEN
8,PART_1A_BASE,output_heatmap_resolution,SAME,False,SUPERVISOR_FROZEN
9,PART_1A_BASE,target_sigma,SAME,False,SUPERVISOR_FROZEN


rows: 32 | amendments: {'PART_1A_BASE': 27, 'A': 1, 'B': 1, 'C': 1, 'D': 1, 'E': 1}
unauthorized material differences: 0


In [10]:
amd = diff[diff['amendment_id'] != 'PART_1A_BASE']
display(amd[['amendment_id', 'field', 'historical_v2', 'proposed_clean_v3', 'authorization']])


,amendment_id,field,historical_v2,proposed_clean_v3,authorization
27,A,A_manifest_population,phase4_v2 / FULL_FIELD VAL 106 images / 156 components,phase4_v3 / FULL_FIELD VAL 103 images / 152 components,AUTHORIZED
28,B,B_hard_set_membership,16 hard-Tiny component IDs + 7 global-empty image IDs (phase4_v2),re-derived on clean-v3 using the SAME historical definition (15 hard-Tiny components /...,AUTHORIZED
29,C,C_inference_heatmap_resolution_contract,implicit / defective generic path (learned_roi.predict_localizer omits heatmap_resolut...,explicit 128x128 fail-loud contract on every clean-v3 path,AUTHORIZED
30,D,D_proposal_grid_selection,"per-epoch best-of-grid over the 20-config proposal grid, used inside the checkpoint score",fixed historically selected configuration 0.10 / 0.20 / 5 / 0.30; per-epoch grid searc...,AUTHORIZED
31,E,E_phase14_crop_distribution_issue,Phase-14 refiner trained on GT-derived/perturbed/negative crops but evaluated on predi...,NOT a Phase-13 localizer change; deferred to post-gate Phase-14 governance,DEFERRED_TO_PHASE14_GOVERNANCE


## 8. Phase-14 distribution requirement (deferred)

The verified historical mismatch is **not** a Phase-13 localizer training change. Any wording
implying "predicted-proposal training crops" belong to Phase-13 Part-1B localizer training is
removed.


In [11]:
p14 = r1.PHASE14_DISTRIBUTION_REQUIREMENT
print('disposition:', p14['canonical_disposition'])
print('is a Phase-13 localizer training change:', p14['is_a_phase13_localizer_training_change'])
print('Phase 13 trains:', p14['phase13_trains'])
print('predicted-proposal training crops in Phase 13:', p14['phase13_predicted_proposal_training_crops'])
print('Phase-14 crop recipe frozen here:', p14['phase14_crop_training_recipe_frozen_here'])
print('Phase 14 started:', p14['phase14_started'])
print()
print('requirement:', p14['requirement'])


disposition: DEFERRED_REQUIREMENT_FOR_PHASE14_IF_GATE_ALLOWS
is a Phase-13 localizer training change: False
Phase 13 trains: image -> localization heatmap / proposals (NOT a regional segmentation refiner)
predicted-proposal training crops in Phase 13: False
Phase-14 crop recipe frozen here: False
Phase 14 started: False

requirement: If the post-Phase-13 gate permits Phase 14, its regional-refiner training distribution MUST be prospectively designed to match, or explicitly model, the frozen predicted-proposal distribution.


## 9. Downstream predicted-region contract (frozen, NOT executed)

Label: **`PRACTICAL_GT_FREE_ORACLE_BENEFIT_RETENTION`** — the retention experiment includes
localization imperfection **and** the practical proposal-geometry difference, so it is not a
pure localization-error isolation study.


In [12]:
g = r1.DOWNSTREAM_GEOMETRY
print('geometry:', g['config'])
print('must NOT change margin to the Phase-12 oracle margin', g['must_not_change_margin_to_phase12_oracle_margin'])
print('retention includes:', ', '.join(g['retention_includes']))
print('pure localization-error isolation study:', not g['not_a_pure_localization_error_isolation_study'])
print('label:', g['label'])
print()
c = r1.DOWNSTREAM_COMPOSITION
print('primary name :', c['primary_name'])
print('  regional crop resolution       :', c['regional_crop_resolution'])
print('  regional prob restoration      :', c['regional_probability_restoration'])
print('  multiple regional crops        :', c['multiple_predicted_regional_crops'])
print('  outside proposal union         :', c['global_outside_predicted_proposal_union'])
print('  inside proposal support        :', c['inside_predicted_proposal_support'])
print('  final threshold (full native)  :', c['final_threshold_full_native_probability'])
print('  FORBIDDEN                      :', ', '.join(c['forbidden']))
print()
print('downstream model scope:', ', '.join(r1.DOWNSTREAM_MODEL_SCOPE))
print('downstream reference model:', r1.DOWNSTREAM_REFERENCE_MODEL)


geometry: {'threshold': 0.1, 'margin': 0.2, 'top_k': 5, 'nms_iou': 0.3}
must NOT change margin to the Phase-12 oracle margin 0.25
retention includes: localization imperfection, practical proposal geometry difference
pure localization-error isolation study: False
label: PRACTICAL_GT_FREE_ORACLE_BENEFIT_RETENTION

primary name : PREDICTED_REGION_FLOAT_GLOBAL_RETAINED
  regional crop resolution       : 512x512
  regional prob restoration      : FLOAT_BILINEAR
  multiple regional crops        : pixelwise MAX among REGIONAL probabilities
  outside proposal union         : RETAIN GLOBAL probability
  inside proposal support        : REPLACE with REGIONAL probability
  final threshold (full native)  : 0.5
  FORBIDDEN                      : max(global, regional) inside proposal support, averaging global/regional, learned fusion

downstream model scope: UNet, Attention U-Net, UNet++, DeepLabV3+
downstream reference model: None


## 10. Retention metric contract and gate governance


In [13]:
rc = r1.RETENTION_CONTRACT
for k, v in rc['aggregate_model_level'].items():
    print(f'  {k}: {v}')
print('  numerical sufficient-retention threshold:', rc['numerical_sufficient_retention_threshold'])
print('  case level:', rc['case_level'])
print()
gg = r1.GATE_GOVERNANCE
print('dimension B:', gg['dimension_b'], '->', gg['dimension_b_status'])
print('scalar gate score:', gg['scalar_gate_score'])
print('numerical GO threshold:', gg['numerical_go_threshold'])
print('final regional gate:', gg['final_regional_gate'])
print('outcomes issued here:', gg['outcomes_issued_in_this_task'] or 'NONE')


  oracle_gain_dice: ORACLE - GLOBAL
  predicted_gain_dice: PREDICTED - GLOBAL
  oracle_gap_retention_dice: predicted_gain_dice / oracle_gain_dice (only where oracle_gain_dice > 0)
  object_recall: same construction
  numerical sufficient-retention threshold: None
  case level: report raw deltas; do NOT force ratios where oracle delta <= 0

dimension B: PRACTICAL_LOCALIZATION_RETENTION -> CANDIDATE_FOR_SUPERVISOR_FREEZE (final evidence contract recorded by this R1)
scalar gate score: None
numerical GO threshold: None
final regional gate: NOT_DECIDED
outcomes issued here: NONE


## 11. Label defects recorded (no historical artifact modified)


In [14]:
for name, d in r1.LABEL_DEFECTS.items():
    print(f'--- {name}')
    for k, v in d.items():
        print(f'    {k}: {v}')
    print()


--- STAGE4D_DESIGN_AUDIT_LABEL_PROVENANCE_DEFECT
    artifact: reports/audits/phase13b_stage4d_design_audit.json
    label: clean_v3_results
    status: NOT_ACCEPTED_AS_CLEAN_V3_EVIDENCE
    evidence: the denominator 156 components matches historical phase4_v2
    disposition: HISTORICAL_v2_PROVENANCE_ONLY
    do_not_reuse_as_clean_v3_metrics: True
    historical_artifact_modified: False

--- NON_BLOCKING_HISTORICAL_ARTIFACT_GAP
    artifact: experiments/phase13b_kub_localizer_r512_h128_sigmaeq075_centernetfocal_seed42_v1/val_search_epoch079.csv
    status: MISSING
    existing_search_dumps: 87
    total_epochs_in_history: 88
    history_contiguous: True
    disposition: NON_BLOCKING_HISTORICAL_ARTIFACT_GAP
    recreated: False
    fabricated: False
    history_modified: False
    authoritative_epoch_provenance: history.csv



## 12. Zero training / inference boundary

Explicit confirmation that this amendment executed no scientific computation.


In [15]:
gv = r1.GOVERNANCE
for k, v in gv.items():
    print(f'  {k}: {v}')
assert gv['training'] == 0 and gv['optimizer_steps'] == 0 and gv['model_forward_passes'] == 0
assert gv['localizer_inference'] == 0 and gv['predicted_region_inference'] == 0
assert gv['proposals_generated'] == 0 and gv['test_content_reads'] == 0
assert not gv['phase14_started'] and not gv['historical_artifacts_modified']
print()
print('BOUNDARY VERIFIED: 0 training, 0 inference, 0 proposals, 0 TEST reads, 0 historical edits')


  training: 0
  optimizer_steps: 0
  model_forward_passes: 0
  localizer_inference: 0
  predicted_region_inference: 0
  proposals_generated: 0
  clean_v3_grid_search_run: False
  test_content_reads: 0
  test_metrics: NONE
  phase14_started: False
  external_data_used: False
  experiment_a_modified: False
  winner_declared: False
  reference_model_selected: False
  regional_mechanism_promoted: False
  historical_artifacts_modified: False

BOUNDARY VERIFIED: 0 training, 0 inference, 0 proposals, 0 TEST reads, 0 historical edits


## 13. Epoch provenance (reference only — not duplicated)

- Authoritative epoch provenance: `history.csv` — **88 / 88 contiguous epochs**.
- The accepted Part-1A notebook already contains the complete upstream epoch tables
  (59 / 69 / 85 / 67 for UNet / Attention U-Net / UNet++ / DeepLabV3+); they are **not**
  duplicated here.
- `val_search_epoch079.csv` remains absent (87 of 88 dumps) — **not recreated, not fabricated**.


In [16]:
hist = PROJECT_ROOT / 'experiments/phase13b_kub_localizer_r512_h128_sigmaeq075_centernetfocal_seed42_v1/history.csv'
h = pd.read_csv(hist)
eps = h['epoch'].astype(int).tolist()
print('history.csv rows:', len(h), '| epochs:', min(eps), '->', max(eps),
      '| contiguous:', eps == list(range(1, len(h) + 1)))
print('history SHA:', r1.sha256_file(hist))
sd = hist.parent
dumps = sorted(p.name for p in sd.glob('val_search_epoch*.csv'))
print('val_search dumps present:', len(dumps), '| epoch079 present:', 'val_search_epoch079.csv' in dumps)
print('part1a notebook contains the 59/69/85/67 tables:', True, '(verified unchanged above)')


history.csv rows: 88 | epochs: 1 -> 88 | contiguous: True
history SHA: dcdb4369d0c3a087fd0bd2611db0aac3ff0afd48db8c1f9a9bb5e79b834fdb76
val_search dumps present: 87 | epoch079 present: False
part1a notebook contains the 59/69/85/67 tables: True (verified unchanged above)


## 14. Frozen protocol artifact


In [17]:
frozen = json.loads(r1.FROZEN_JSON.read_text(encoding='utf-8'))
print('status :', frozen['status'])
print('json   :', r1.FROZEN_JSON.name, '->', hashes[r1.FROZEN_JSON.name])
print('md     :', r1.FROZEN_MD.name, '->', hashes[r1.FROZEN_MD.name])
print()
print('pass gate:')
for k, v in frozen['pass_gate'].items():
    print(f'  {k}: {v}')
# NOTE: 'unauthorized_material_differences' is a COUNT whose required value is 0
# (zero unauthorized differences), so it must be checked for equality, not truthiness.
assert frozen['pass_gate']['unauthorized_material_differences'] == 0
assert all(v for k, v in frozen['pass_gate'].items()
           if k != 'unauthorized_material_differences')
print()
print('ALL PASS-GATE ITEMS TRUE (unauthorized material differences == 0)')


status : SUPERVISOR_FROZEN_READY_FOR_PART1B_TRAINING
json   : phase13_clean_v3_localizer_protocol_frozen.json -> 999144bbad4af4e1d2dc006279aef1a5fb93f450006ae2c5db159c791c913703
md     : phase13_clean_v3_localizer_protocol_frozen.md -> f62a562ae68091d83d6428e9b5ed6b0f1e37256950e00d2053f60d5fda2b27f0

pass gate:
  hard_sets_rederived_and_frozen_before_training: True
  historical_definitions_preserved: True
  unetpp_role_replication_anchor_only: True
  heatmap_output_contract_128: True
  shape_assertions_specified: True
  stage4d_label_ambiguity_resolved_as_provenance_defect: True
  missing_epoch079_unfabricated: True
  primary_proposal_config_fixed: True
  historical_20_grid_not_used_for_primary_selection: True
  fixed_config_lexicographic_checkpoint_metric_frozen: True
  predicted_proposal_training_crops_removed_from_phase13: True
  phase14_distribution_mismatch_retained_as_deferred_requirement: True
  future_predicted_region_practical_geometry_frozen: True
  future_downstream_global_r

In [18]:
reg = pd.DataFrame([{'artifact': k, 'sha256': v} for k, v in sorted(hashes.items())])
display(reg)
print()
print('NEXT ACTION:', frozen['next_action'])


,artifact,sha256
0,phase13_clean_v3_global_empty_replication_set.csv,a0b332cad6bca4466ccc528cbe18cc527ef6503603ab69c55b9d699d7979f0ea
1,phase13_clean_v3_hard_tiny_replication_set.csv,2f9828d77c008e7c90cd69e82d0af3568902ca9d508fdfcb9d8ccf8abd569538
2,phase13_clean_v3_localizer_protocol_frozen.json,999144bbad4af4e1d2dc006279aef1a5fb93f450006ae2c5db159c791c913703
3,phase13_clean_v3_localizer_protocol_frozen.md,f62a562ae68091d83d6428e9b5ed6b0f1e37256950e00d2053f60d5fda2b27f0
4,phase13_v2_to_clean_v3_localizer_protocol_diff.csv,e701b8885e040991a977a88f4a4972d6573cd74620430a4634b71528e2b08f09



NEXT ACTION: WAIT_FOR_SUPERVISOR_PHASE13_PART1B_TRAINING_AUTHORIZATION


## 15. STOP

- Part 1A archaeology: **ACCEPTED**. Part 1A protocol: **frozen** at `SUPERVISOR_FROZEN_READY_FOR_PART1B_TRAINING`.
- Part 1B: **ELIGIBLE / NOT AUTHORIZED**.
- `REGIONAL_INTEGRATION_GATE_AFTER_PHASE13`: **NOT_DECIDED** (no GO / REVISE_LOCALIZER / NO_GO issued).
- Phase 14: **not started**. `RA-MedFiLM-Seg++`: **not implemented**.
- TEST: **untouched** (content reads 0).

**STOP — WAIT_FOR_SUPERVISOR_PHASE13_PART1B_TRAINING_AUTHORIZATION.**


In [19]:
print('PHASE 13 PART 1A-R1 — COMPLETE (supervisor freeze amendment)')
print('protocol status: SUPERVISOR_FROZEN_READY_FOR_PART1B_TRAINING')
print('part1b: ELIGIBLE / NOT AUTHORIZED')
print('gate: NOT_DECIDED | phase14: not started | RA-MedFiLM: not implemented')
print('training 0 | inference 0 | proposals 0 | TEST reads 0')


PHASE 13 PART 1A-R1 — COMPLETE (supervisor freeze amendment)
protocol status: SUPERVISOR_FROZEN_READY_FOR_PART1B_TRAINING
part1b: ELIGIBLE / NOT AUTHORIZED
gate: NOT_DECIDED | phase14: not started | RA-MedFiLM: not implemented
training 0 | inference 0 | proposals 0 | TEST reads 0
